<a href="https://colab.research.google.com/github/suniltejh361-arch/dataviz-exercises-suniltejh361/blob/main/lecture03_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 3 — Class Exercise
## Line Charts & Slopegraphs: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
1. No spaghetti — multiple lines must use grey + single highlight
2. Remove clutter: no chart borders, no heavy gridlines, no legend if you can label directly
3. Insight title — states the finding, not the topic
4. Carry forward from Lecture 2: white background, Arial font, professional quality


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('/content/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [2]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


---
## Task 1 — Multi-Series Line Chart with Highlight

**What to build:** A line chart showing CO2 emissions over time for **all Asian countries** in the dataset, with one country highlighted.

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#DDDDDD), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

> 💡 `df[df['Region'] == 'Asia']` to filter; use `go.Figure()` with a loop for per-country control


In [3]:
# Task 1 — Multi-series line with highlight

# Filter Asia data
asia_df = df[df['Region'] == 'Asia']

highlight_country = 'India'  # you can change this

fig = go.Figure()

# Add all countries in grey
for country in asia_df['Country'].unique():
    country_data = asia_df[asia_df['Country'] == country]

    if country == highlight_country:
        fig.add_trace(go.Scatter(
            x=country_data['Year'],
            y=country_data['CO2_Mt'],
            mode='lines',
            name=country,
            line=dict(width=3)  # highlight
        ))
    else:
        fig.add_trace(go.Scatter(
            x=country_data['Year'],
            y=country_data['CO2_Mt'],
            mode='lines',
            line=dict(color='lightgrey', width=1),
            showlegend=False
        ))

# Clean layout
fig.update_layout(
    title=f"CO2 Emissions in Asia (Highlight: {highlight_country})",
    xaxis_title="Year",
    yaxis_title="CO2 Emissions (Mt)",
    template="simple_white"
)

fig.show()


---
## Task 2 — Slopegraph: Regional Change 2000 vs 2022

**What to build:** A slopegraph comparing **average regional CO2 emissions** between 2000 and 2022.

**Requirements:**
- One line per region (not per country — aggregate first)
- Colour: regions that increased = one colour; decreased = another
- Values labelled at both ends of each line
- No y-axis tick labels (the endpoint labels make them redundant)
- Insight title stating which regions moved most

> 💡 `df.groupby(['Region','Year'])['CO2_Mt'].mean().reset_index()` then filter to 2000 and 2022


In [4]:
# Task 2 — Slopegraph: regional averages

# Aggregate data
df_2000 = df[df['Year'] == 2000].groupby('Region')['CO2_Mt'].mean()
df_2022 = df[df['Year'] == 2022].groupby('Region')['CO2_Mt'].mean()

regions = df_2000.index

fig = go.Figure()

for region in regions:
    y0 = df_2000[region]
    y1 = df_2022[region]

    color = 'red' if y1 > y0 else 'green'

    fig.add_trace(go.Scatter(
        x=[0, 1],
        y=[y0, y1],
        mode='lines+markers+text',
        line=dict(color=color, width=2),
        text=[f"{region} ({round(y0,1)})", f"{round(y1,1)}"],
        textposition="top center",
        showlegend=False
    ))

# Clean layout
fig.update_layout(
    title="Regional CO2 Emissions Change (2000 → 2022)",
    xaxis=dict(
        tickvals=[0, 1],
        ticktext=["2000", "2022"]
    ),
    yaxis_title="Avg CO2 Emissions (Mt)",
    template="simple_white"
)

fig.show()

CO₂ emissions have increased significantly over time, especially in Asia and developing regions. Growth is uneven across countries, but the overall global trend is upward. Emerging economies are the main contributors to this rise, highlighting the need for stronger sustainability efforts and cleaner energy transitions.